### Price prediction model

#### Imports and Configuration

This notebook implements a regression model using the DataFrame-based Spark ML API (`pyspark.ml`).

A Gradient-Boosted Tree (GBT) Regressor is selected because it can model nonlinear relationships between listing characteristics and advertised prices without requiring feature scaling. The model is trained using listings from both cities.

The notebook uses a fixed random seed to make the train/test split reproducible. The target variable is the nightly listing price in USD. Because listing prices are right-skewed, the model is trained on the log-transformed price, and predictions are transformed back to USD for evaluation.


In [0]:

from pyspark.sql import functions as F

from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.regression import GBTRegressor

CATALOG = spark.sql(
    "SELECT current_catalog()"
).first()[0]

SCHEMA = "default"

# Table configuration
GOLD_FEATURES_TABLE = f"{CATALOG}.{SCHEMA}.gold_listings_features"
GOLD_PREDICTIONS_TABLE = f"{CATALOG}.{SCHEMA}.gold_price_predictions"
GOLD_METRICS_TABLE = f"{CATALOG}.{SCHEMA}.gold_model_metrics"

# Experiment configuration
RANDOM_SEED = 67
MODEL_VERSION = "airbnb_v1"

print(f"Input table: {GOLD_FEATURES_TABLE}")
print(f"Predictions table: {GOLD_PREDICTIONS_TABLE}")
print(f"Metrics table: {GOLD_METRICS_TABLE}")


Input table: workspace.default.gold_listings_features
Predictions table: workspace.default.gold_price_predictions
Metrics table: workspace.default.gold_model_metrics



#### Loading the Gold Feature Dataset

The model uses the `gold_listings_features` table produced by the ETL pipeline.

Each row represents one Airbnb listing at the selected snapshot date. The `price` column is the target variable, representing the advertised price per night.

The listing identifier and host identifier are retained for traceability but are not used as model features. The price column is also excluded from the feature vector to prevent target leakage.

Only records with a valid positive price can be used for supervised training and evaluation.


In [0]:

source_df = spark.table(GOLD_FEATURES_TABLE)

# Check that the required columns are present.
required_columns = [
    "id",
    "city",
    "snapshot_date",
    "price",
    "room_type",
    "accommodates",
    "property_type",
    "neighbourhood_cleansed"
]

missing_columns = [
    c for c in required_columns
    if c not in source_df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

# The ETL pipeline should already have removed invalid prices.
# Validate this assumption rather than silently changing the dataset.
invalid_price_count = source_df.filter(
    F.col("price").isNull() |
    (F.col("price") <= 0)
).count()

if invalid_price_count > 0:
    raise ValueError(
        f"Gold contains {invalid_price_count} listings with invalid prices. "
        "Check the Silver and Gold ETL transformations."
    )

# Convert the target and numeric/boolean features to consistent types.
source_df = source_df.withColumn("price", F.col("price").cast("double"))

boolean_columns = [
    "host_is_superhost",
    "bathroom_shared",
    "is_studio",
    "has_reviews"
]

for c in boolean_columns:
    if c in source_df.columns:
        source_df = source_df.withColumn(c, F.col(c).cast("double"))

categorical_columns = [
    "city",
    "neighbourhood_cleansed",
    "property_type",
    "room_type"
]

for c in categorical_columns:
    source_df = source_df.withColumn(c, F.col(c).cast("string"))

cities = [
    row["city"]
    for row in source_df.select("city").distinct().collect()
]

if len(cities) != 2:
    raise ValueError(
        f"Expected exactly two training cities, but found: {cities}"
    )

print(f"Cities included in training: {cities}")
print(f"Total listings: {source_df.count():,}")

display(
    source_df.groupBy("city")
    .agg(
        F.count("*").alias("listing_count"),
        F.avg("price").alias("average_price"),
        F.expr("percentile_approx(price, 0.5)").alias("median_price")
    )
)


Cities included in training: ['austin', 'fort_worth']
Total listings: 11,910


city,listing_count,average_price,median_price
austin,9759,298.54772620145576,225.85
fort_worth,2151,207.86589493258936,167.5



#### Train/Test Split

The dataset is divided into training and test subsets before calculating baseline prices, imputing missing numerical values, or fitting any machine learning transformations.

A separate random 80/20 split is performed for each city. This ensures that both Austin and Fort Worth are represented in the training and test datasets.

The fixed random seed makes the split reproducible under the same execution conditions.

The test subset is reserved exclusively for final model and baseline evaluation. It is not used to calculate feature statistics or train the model.


In [0]:
train_parts = []
test_parts = []

for city_name in cities:
    city_df = source_df.filter(F.col("city") == city_name)

    city_train, city_test = city_df.randomSplit(
        [0.8, 0.2],
        seed=RANDOM_SEED
    )

    train_parts.append(
        city_train.withColumn("split", F.lit("train"))
    )

    test_parts.append(
        city_test.withColumn("split", F.lit("test"))
    )

# Combine the splits from both cities
train_df = train_parts[0]
test_df = test_parts[0]

for part in train_parts[1:]:
    train_df = train_df.unionByName(part)

for part in test_parts[1:]:
    test_df = test_df.unionByName(part)

# Combine training and test data for later predictions
all_df = train_df.unionByName(test_df)

print(f"Training listings: {train_df.count():,}")
print(f"Test listings: {test_df.count():,}")
print(f"Total listings: {all_df.count():,}")

display(
    all_df.groupBy("city", "split")
    .count()
    .orderBy("city", "split")
)

Training listings: 9,474
Test listings: 2,436
Total listings: 11,910


city,split,count
austin,test,1997
austin,train,7762
fort_worth,test,439
fort_worth,train,1712


#### Prepare numerical features and handle missing values

In [0]:
numeric_features = [
    "latitude",
    "longitude",
    "accommodates",
    "bathrooms",
    "bedrooms",
    "beds",
    "calculated_host_listings_count",
    "minimum_nights",
    "maximum_nights",
    "availability_30",
    "availability_365",
    "estimated_occupancy_l365d",
    "number_of_reviews",
    "review_scores_rating",
    "review_scores_cleanliness",
    "review_scores_location",
    "reviews_per_month",
    "host_is_superhost",
    "bathroom_shared",
    "is_studio",
    "has_reviews"
]

# Keep only numerical features available in the Gold table
numeric_features = [
    c for c in numeric_features
    if c in train_df.columns
]

# Cast numerical features to double
for c in numeric_features:
    train_df = train_df.withColumn(c, F.col(c).cast("double"))
    test_df = test_df.withColumn(c, F.col(c).cast("double"))
    all_df = all_df.withColumn(c, F.col(c).cast("double"))

# Calculate medians using training data only
median_expressions = [
    F.expr(f"percentile_approx(`{c}`, 0.5)").alias(c)
    for c in numeric_features
]

training_medians = train_df.agg(
    *median_expressions
).first().asDict()

median_fill_values = {
    c: float(training_medians[c])
    if training_medians[c] is not None
    else 0.0
    for c in numeric_features
}

# Apply the same training-derived medians to all datasets
train_model_df = train_df.fillna(median_fill_values)
test_model_df = test_df.fillna(median_fill_values)
all_model_df = all_df.fillna(median_fill_values)

print("Numerical features:", numeric_features)
print("Missing numerical values handled.")

Numerical features: ['latitude', 'longitude', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'calculated_host_listings_count', 'minimum_nights', 'maximum_nights', 'availability_30', 'availability_365', 'estimated_occupancy_l365d', 'number_of_reviews', 'review_scores_rating', 'review_scores_cleanliness', 'review_scores_location', 'reviews_per_month', 'host_is_superhost', 'bathroom_shared', 'is_studio', 'has_reviews']
Missing numerical values handled.


#### Create the median baseline

In [0]:
def prepare_baseline_keys(df):
    return (
        df
        .withColumn(
            "_baseline_room_type",
            F.coalesce(F.col("room_type"), F.lit("Unknown"))
        )
        .withColumn(
            "_baseline_accommodates",
            F.coalesce(
                F.col("accommodates").cast("int"),
                F.lit(-1)
            )
        )
    )


# Calculate group medians using training data only
baseline_training_df = prepare_baseline_keys(train_df)

baseline_lookup = (
    baseline_training_df
    .groupBy(
        "_baseline_room_type",
        "_baseline_accommodates"
    )
    .agg(
        F.expr("percentile_approx(price, 0.5)")
        .alias("group_median_price")
    )
)

# Overall training median for unseen combinations
overall_training_median = train_df.approxQuantile(
    "price",
    [0.5],
    0.001
)[0]


def attach_baseline(df):
    keyed_df = prepare_baseline_keys(df)

    return (
        keyed_df
        .join(
            baseline_lookup,
            on=[
                "_baseline_room_type",
                "_baseline_accommodates"
            ],
            how="left"
        )
        .withColumn(
            "baseline_prediction",
            F.coalesce(
                F.col("group_median_price"),
                F.lit(float(overall_training_median))
            )
        )
        .drop(
            "_baseline_room_type",
            "_baseline_accommodates",
            "group_median_price"
        )
    )


print(
    f"Overall training median: "
    f"${overall_training_median:.2f}"
)

Overall training median: $215.00


#### Define and train the GBT model
Categorical features are converted using StringIndexer and OneHotEncoder. VectorAssembler combines the numerical and encoded categorical features into the feature vector.

The model is trained on the log-transformed price (`log1p(price)`) to reduce the skew of the price distribution. The fitted model therefore predicts log prices, which are transformed back to USD with the inverse transform (`expm1`) before any evaluation or output.

The pipeline is fitted only on the training dataset.

In [0]:
categorical_features = [
    "city",
    "neighbourhood_cleansed",
    "property_type",
    "room_type"
]

pipeline_stages = []
encoded_features = []

for c in categorical_features:
    index_col = f"{c}_index"
    encoded_col = f"{c}_encoded"

    indexer = StringIndexer(
        inputCol=c,
        outputCol=index_col,
        handleInvalid="keep"
    )

    encoder = OneHotEncoder(
        inputCol=index_col,
        outputCol=encoded_col,
        handleInvalid="keep"
    )

    pipeline_stages.extend([indexer, encoder])
    encoded_features.append(encoded_col)


assembler = VectorAssembler(
    inputCols=numeric_features + encoded_features,
    outputCol="features",
    handleInvalid="error"
)

# Train on the log-transformed price to reduce skew;
# predictions are transformed back to USD with expm1 below.
train_model_df = train_model_df.withColumn(
    "log_price", F.log1p(F.col("price"))
)
test_model_df = test_model_df.withColumn(
    "log_price", F.log1p(F.col("price"))
)
all_model_df = all_model_df.withColumn(
    "log_price", F.log1p(F.col("price"))
)

gbt = GBTRegressor(
    featuresCol="features",
    labelCol="log_price",
    predictionCol="model_prediction",
    maxIter=100,
    maxDepth=5,
    seed=RANDOM_SEED
)

price_pipeline = Pipeline(
    stages=pipeline_stages + [
        assembler,
        gbt
    ]
)

# Fit the complete pipeline using training data only
price_model = price_pipeline.fit(train_model_df)

print("GBT model trained successfully.")

GBT model trained successfully.


#### Generate test predictions and calculate MAE
Both the model and baseline are evaluated using the same holdout test listings. The GBT predicts log prices, so its raw output is transformed back to USD before the MAE is calculated.

In [0]:
# Generate predictions for the test dataset
# The model predicts log1p(price); transform back to USD
test_predictions = price_model.transform(test_model_df).withColumn(
    "model_prediction",
    F.expm1(F.col("model_prediction"))
)

test_baseline = (
    attach_baseline(test_df)
    .select(
        "id",
        "city",
        "snapshot_date",
        "baseline_prediction"
    )
)

test_scored = (
    test_predictions
    .join(
        test_baseline,
        on=["id", "city", "snapshot_date"],
        how="inner"
    )
)

def calculate_mae(df, prediction_column):
    result = df.agg(
        F.avg(
            F.abs(
                F.col("price") - F.col(prediction_column)
            )
        ).alias("mae")
    ).first()

    return float(result["mae"])


metric_rows = []

evaluation_groups = [
    (
        city_name,
        test_scored.filter(F.col("city") == city_name)
    )
    for city_name in cities
]

# Add combined results across both cities
evaluation_groups.append(("combined", test_scored))


for city_name, evaluation_df in evaluation_groups:
    test_count = evaluation_df.count()

    if test_count == 0:
        raise ValueError(
            f"No test listings found for {city_name}"
        )

    model_mae = calculate_mae(
        evaluation_df,
        "model_prediction"
    )

    baseline_mae = calculate_mae(
        evaluation_df,
        "baseline_prediction"
    )

    metric_rows.append((
        city_name,
        MODEL_VERSION,
        "GBTRegressor",
        model_mae,
        test_count,
        "test",
        "USD"
    ))

    metric_rows.append((
        city_name,
        MODEL_VERSION,
        "training_median_baseline",
        baseline_mae,
        test_count,
        "test",
        "USD"
    ))


metrics_df = spark.createDataFrame(
    metric_rows,
    [
        "city",
        "model_version",
        "model_type",
        "mae",
        "test_count",
        "dataset_split",
        "currency"
    ]
).withColumn(
    "evaluation_timestamp",
    F.current_timestamp()
)

# Save evaluation metrics
(
    metrics_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_METRICS_TABLE)
)

display(
    metrics_df.orderBy("city", "model_type")
)

city,model_version,model_type,mae,test_count,dataset_split,currency,evaluation_timestamp
austin,airbnb_v1,GBTRegressor,79.7511157322429,1997,test,USD,2026-11-11T21:43:02.087Z
austin,airbnb_v1,training_median_baseline,111.42428642964447,1997,test,USD,2026-11-11T21:43:02.087Z
combined,airbnb_v1,GBTRegressor,76.27978350892695,2436,test,USD,2026-11-11T21:43:02.087Z
combined,airbnb_v1,training_median_baseline,107.9690763546798,2436,test,USD,2026-11-11T21:43:02.087Z
fort_worth,airbnb_v1,GBTRegressor,60.48878020605247,439,test,USD,2026-11-11T21:43:02.087Z
fort_worth,airbnb_v1,training_median_baseline,92.25141230068331,439,test,USD,2026-11-11T21:43:02.087Z


We will visualize the baseline and model predictions vs actual listing prices on both cities with scatter plots:

In [0]:
actual_vs_predicted_austin = (
    test_scored
    .filter((F.col("city") == "austin") & (F.col("price") <= 100) & (F.col("price") >= 50))
    .select(
        "id",
        "price",
        "baseline_prediction",
        "model_prediction"
    )
    .orderBy(F.col("price").asc())
)

display(actual_vs_predicted_austin)

id,price,baseline_prediction,model_prediction
1449301332968456081,50.5,143.26,96.19340165248393
26008727,50.52,198.0,78.70079142025844
50344928,52.0,49.0,50.78857924157977
1492763750836454338,52.0,143.26,67.08192493502165
725995552075662828,52.0,93.05,53.729792938709984
1028296374473223363,52.23,93.05,67.07211499324066
551013304459329499,52.46,49.0,134.23626023544637
1280710848136631636,53.0,93.05,78.30499918650071
1280718313866356892,53.0,93.05,80.0434000661414
51111910,53.14,49.0,35.714009526137694


Databricks visualization. Run in Databricks to view.

In [0]:
actual_vs_predicted_fortworth = (
    test_scored
    .filter((F.col("city") == "fort_worth") & (F.col("price") <= 100) & (F.col("price") >= 50))
    .select(
        "id",
        "price",
        "baseline_prediction",
        "model_prediction"
    )
    .orderBy(F.col("price").asc())
)

display(actual_vs_predicted_fortworth)

id,price,baseline_prediction,model_prediction
963128046112137577,50.0,93.05,86.81178780474535
1718090446158595224,50.0,93.05,78.54351309534879
28995548,50.5,93.05,61.536151063153206
28992686,50.5,93.05,58.597036664177054
1735490410134112677,50.78,143.26,77.97427837522375
1751733253990120192,51.06,49.0,54.51884216629346
1700152153011734773,52.0,49.0,64.32856160346931
1627225740178177421,54.0,93.05,50.33540416051889
974290290833111885,54.06,93.05,90.41457393891791
45310670,54.33,143.26,80.81291673677948


Databricks visualization. Run in Databricks to view.

#### Generate predictions for all listings

In [0]:
# Generate model predictions for all listings
# The model predicts log1p(price); transform back to USD
all_predictions = price_model.transform(all_model_df).withColumn(
    "model_prediction",
    F.expm1(F.col("model_prediction"))
)

# Add baseline predictions
all_scored = attach_baseline(all_predictions)

print(f"Scored listings: {all_scored.count():,}")

display(
    all_scored.select(
        "id",
        "city",
        "split",
        "price",
        "model_prediction",
        "baseline_prediction"
    ).limit(10)
)

Scored listings: 11,910


id,city,split,price,model_prediction,baseline_prediction
5456,austin,train,135.5,152.1077660628188,160.0
6448,austin,train,242.33,214.07243613865285,143.26
13035,austin,train,127.5,128.52457234244713,160.0
22828,austin,train,65.89,72.04084259320157,143.26
50334,austin,train,300.0,246.35424383800984,143.26
57187,austin,train,176.31,180.50632329124045,266.25
69303,austin,train,523.35,255.24561981771384,198.0
73289,austin,train,208.26,283.93987668861837,198.0
76501,austin,train,168.0,140.49532817062297,160.0
77639,austin,train,185.33,312.5203117255637,198.0


#### Save predictions to Gold and validate

In [0]:
# Select the columns required for the dashboard and Genie
predictions_output = all_scored.select(
    "id",
    "city",
    "snapshot_date",
    "split",
    "room_type",
    "accommodates",
    F.col("price").alias("actual_price"),
    F.col("model_prediction").alias("predicted_price"),
    F.col("baseline_prediction").alias("baseline_price"),
    F.abs(
        F.col("price") - F.col("model_prediction")
    ).alias("model_absolute_error"),
    F.abs(
        F.col("price") - F.col("baseline_prediction")
    ).alias("baseline_absolute_error"),
    F.lit(MODEL_VERSION).alias("model_version"),
    F.current_timestamp().alias("prediction_timestamp")
)

# Write predictions to the Gold table
if spark.catalog.tableExists(GOLD_PREDICTIONS_TABLE):
    (
        predictions_output.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(GOLD_PREDICTIONS_TABLE)
    )
else:
    (
        predictions_output.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("city", "snapshot_date")
        .saveAsTable(GOLD_PREDICTIONS_TABLE)
    )

print(f"Predictions saved to {GOLD_PREDICTIONS_TABLE}")


# Validate saved predictions
saved_predictions = spark.table(GOLD_PREDICTIONS_TABLE)

source_count = all_df.count()
prediction_count = saved_predictions.count()

assert prediction_count == source_count, (
    f"Prediction count mismatch: "
    f"expected {source_count}, got {prediction_count}"
)

null_prediction_count = saved_predictions.filter(
    F.col("predicted_price").isNull()
    | F.col("baseline_price").isNull()
).count()

assert null_prediction_count == 0, (
    f"Found {null_prediction_count} rows with null predictions"
)

split_counts = (
    saved_predictions
    .groupBy("city", "split")
    .count()
    .orderBy("city", "split")
)

saved_metrics = spark.table(GOLD_METRICS_TABLE)
metrics_count = saved_metrics.count()

assert metrics_count == 6, (
    f"Expected 6 metric rows, got {metrics_count}"
)

print("All validation checks passed.")
print(f"Source listings: {source_count:,}")
print(f"Saved predictions: {prediction_count:,}")
print(f"Null predictions: {null_prediction_count}")

display(split_counts)

display(
    saved_metrics.orderBy("city", "model_type")
)

Predictions saved to workspace.default.gold_price_predictions
All validation checks passed.
Source listings: 11,910
Saved predictions: 11,910
Null predictions: 0


city,split,count
austin,test,1997
austin,train,7762
fort_worth,test,439
fort_worth,train,1712


city,model_version,model_type,mae,test_count,dataset_split,currency,evaluation_timestamp
austin,airbnb_v1,GBTRegressor,79.7511157322429,1997,test,USD,2026-11-11T21:43:01.083Z
austin,airbnb_v1,training_median_baseline,111.42428642964447,1997,test,USD,2026-11-11T21:43:01.083Z
combined,airbnb_v1,GBTRegressor,76.27978350892695,2436,test,USD,2026-11-11T21:43:01.083Z
combined,airbnb_v1,training_median_baseline,107.9690763546798,2436,test,USD,2026-11-11T21:43:01.083Z
fort_worth,airbnb_v1,GBTRegressor,60.48878020605247,439,test,USD,2026-11-11T21:43:01.083Z
fort_worth,airbnb_v1,training_median_baseline,92.25141230068331,439,test,USD,2026-11-11T21:43:01.083Z


#### Conclusion
The Gradient-Boosted Tree (GBT) Regressor, trained on the log-transformed price, outperformed the median-price baseline across both training cities and the combined test dataset. Its MAE was approximately $79.75 for Austin, $60.49 for Fort Worth, and $76.28 for the combined dataset, compared with baseline MAEs of $111.42, $92.25, and $107.97 respectively. This corresponds to an MAE reduction of approximately 28% for Austin, 34% for Fort Worth, and 29% overall, demonstrating that listing characteristics provide useful information for predicting nightly prices.

Nevertheless, the remaining prediction error indicates that the model does not capture all factors influencing Airbnb prices. Its practical accuracy should be interpreted relative to the distribution of listing prices in each city.